# FC 온라인 인벤 크롤링

## 데이터 수집 환경 조성

### 필요 라이브러리 호출

In [18]:
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from openpyxl import Workbook
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from time import sleep
import matplotlib.pyplot as plt
plt.rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] =False

# Warnings 제거
import warnings
warnings.filterwarnings('ignore')

### 데이터 크롤링

In [2]:
# 검색할 선수 이름, 시즌 카드 받아오기
player_search = input("선수 이름을 입력해주세요.")
season_icon = input("시즌 아이콘을 선택해주세요.")

In [3]:
season_icon

'22 UEFA Champions League'

---

- 카드 시즌 이미지 가져오기

In [5]:
res = requests.get('https://fifaonline4.inven.co.kr/dataninfo/player/')
soup = BeautifulSoup(res.text, 'html.parser')

urls = soup.select('.fifa4.value.season.clearfix > label.checkbox > img')
urls[0]

<img src="//static.inven.co.kr/image_2011/site_image/fifaonline4/seasonicon2/season_icon_100.png?v=2401210a" title="ICON The Moment"/>

In [41]:
# 카드 이미지 저장
res = requests.get('https://fifaonline4.inven.co.kr/dataninfo/player/')
soup = BeautifulSoup(res.text, 'html.parser')

urls = soup.select('.fifa4.value.season.clearfix > label.checkbox > img')

for url in urls:
    image_url = 'https:{}'.format(str(url).split('"')[1])
    response = requests.get(image_url)
    filename = 'logo_{}.png'.format(str(url).split('"')[3])
    with open(filename, 'wb+') as f:
        f.write(response.content)

---

In [ ]:
# 드라이버 생성
driver = webdriver.Chrome()
wait= WebDriverWait(driver,3)

# 사이트 접속하기
driver.get('https://fifaonline4.inven.co.kr/dataninfo/player/')
sleep(3)


################### 선수 페이지로 이동하기 ###################
# 시즌 아이콘 선택
season_icon = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, '.fifa4.value.season.clearfix > label.checkbox > div[title="22 UEFA Champions League"]')))
season_icon.click()

# 선수 이름 검색창 클릭
search_box = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, '.fifa4.value.name_search.clearfix > .input')))
search_box.click()

# 선수 이름 입력
name_input = wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR,'.fifa4.value.name_search.clearfix > .input')))
name_input.send_keys(player_search)

# 검색 버튼 클릭
search_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR,'.fifa4.btn_area.clearfix > .btn.search')))
search_btn.click()

# 웹페이지 끝까지 스크롤
# 맨 위로 스크롤링
driver.execute_script('window.scrollTo(0,0)')

# 처음 웹페이지가 로딩 되었을 때, (불러와진) 높이를 가져오기
last_height = driver.execute_script('return document.body.scrollHeight')

# 반복 및 끝까지 스크롤링
while True:
    # 첫 스크롤링 진행
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight)')
    
    # 새로운 페이지가 보일 때까지 대기
    sleep(1)

    # 현재 높이
    new_height = driver.execute_script('return document.body.scrollHeight')

    if new_height == last_height:
        break
        
    last_height = new_height

# 선수 카드 클릭
player_card = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,'.fifa4.player_info.clearfix > a.fifa4.text_area > span.fifa4.name')))
player_card.click()



################### 모든 카드 리뷰 받아오기 ###################
# 리뷰 리스트 생성
comment_list = []

# 웹페이지 끝까지 스크롤
# 맨 위로 스크롤링
driver.execute_script('window.scrollTo(0,0)')

# 처음 웹페이지가 로딩 되었을 때, (불러와진) 높이를 가져오기
last_height = driver.execute_script('return document.body.scrollHeight')

# 반복 및 끝까지 스크롤링
while True:
    # 첫 스크롤링 진행
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight)')
    
    # 새로운 페이지가 보일 때까지 대기
    sleep(1)

    # 현재 높이
    new_height = driver.execute_script('return document.body.scrollHeight')

    if new_height == last_height:
        break
        
    last_height = new_height

# 첫페이지 리뷰 리스트에 추가
comments = driver.find_elements('css selector', 'span.comment')
for comment in comments:
    comment_list.append(comment.text.replace('\n',' '))

# 다음페이지 클릭
next_text = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'span.nexttext')))
next_text.click()

sleep(5)

# 10페이지까지 반복(10페이지 초과하는 리뷰는 집계X)
for i in range(11):
    comments = driver.find_elements('css selector', 'span.comment')
    for comment in comments:
        comment_list.append(comment.text.replace('\n',' '))
    
    sleep(3)
    # 웹페이지 끝까지 스크롤
    # 맨 위로 스크롤링
    driver.execute_script('window.scrollTo(0,0)')

    # 처음 웹페이지가 로딩 되었을 때, (불러와진) 높이를 가져오기
    last_height = driver.execute_script('return document.body.scrollHeight')

    # 반복 및 끝까지 스크롤링
    while True:
        # 첫 스크롤링 진행
        driver.execute_script('window.scrollTo(0, document.body.scrollHeight)')
        
        # 새로운 페이지가 보일 때까지 대기
        sleep(1)

        # 현재 높이
        new_height = driver.execute_script('return document.body.scrollHeight')

        if new_height == last_height:
            break
            
        last_height = new_height

    # 다음 페이지 버튼 클릭
    next_text = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'span.nexttext')))
    next_text.click()
    sleep(5)

    i+=1


################### 선수 정보 스크래핑(BeautifulSoup) ###################

# 선수 정보 페이지(현재페이지) 받아오기
res = requests.get(driver.current_url)

# 드라이버 종료
sleep(5)
driver.quit()

# 선수 정보 받아오기
soup = BeautifulSoup(res.text, 'html.parser')

player_stat = soup.select('ul.fifa4.tooltip_position.clearfix span.score')[0].get_text() # 스탯
player_age = soup.select('ul.fifa4.state.clearfix')[0].get_text().strip().split('\n')[0] # 나이
player_pay = soup.select('ul.fifa4.state.clearfix')[0].get_text().strip().split('\n')[1].split(' ')[1].strip('[').strip(']') # 급여
player_bodyform = soup.select('ul.fifa4.state.clearfix')[0].get_text().strip().split('\n')[2] # 체형
player_backnum = soup.select('ul.fifa4.state.clearfix')[0].get_text().strip().split('\n')[3].split(' ')[2] # 등번호
player_country = soup.select('ul.fifa4.state.clearfix')[0].get_text().strip().split('\n')[7].strip() # 국적
player_class = soup.select('ul.fifa4.state.clearfix')[0].get_text().strip().split('\n')[8] # 클래스

player_info = {
    '능력치' : int(player_stat),
    '나이' : player_age,
    '급여' : int(player_pay),
    '체형' : player_bodyform,
    '등번호' : player_backnum,
    '국적' : player_country,
    '클래스' : player_class
}

# 선수 정보 확인
print(player_info)


"\nplayer_stat = soup.select('ul.fifa4.tooltip_position.clearfix span.score')[0].get_text() # 스탯\nplayer_age = soup.select('ul.fifa4.state.clearfix')[0].get_text().strip().split('\n')[0] # 나이\nplayer_pay = soup.select('ul.fifa4.state.clearfix')[0].get_text().strip().split('\n')[1].split(' ')[1].strip('[').strip(']') # 급여\nplayer_bodyform = soup.select('ul.fifa4.state.clearfix')[0].get_text().strip().split('\n')[2] # 체형\nplayer_backnum = soup.select('ul.fifa4.state.clearfix')[0].get_text().strip().split('\n')[3].split(' ')[2] # 등번호\nplayer_country = soup.select('ul.fifa4.state.clearfix')[0].get_text().strip().split('\n')[7].strip() # 국적\nplayer_class = soup.select('ul.fifa4.state.clearfix')[0].get_text().strip().split('\n')[8] # 클래스\n\nplayer_info = {\n    '능력치' : int(player_stat),\n    '나이' : player_age,\n    '급여' : int(player_pay),\n    '체형' : player_bodyform,\n    '등번호' : player_backnum,\n    '국적' : player_country,\n    '클래스' : player_class\n}\n\n# 선수 정보 확인\nprint(player_info)\n"

In [28]:
comment_list

['금카 아직 현역이다 7조대 가성비 금흥민 감차 치달 퍼터 중거리 미쳤음 체감까지 괜찮고 금카케미까지 받을 수 있어서 메리트 있음 진짜 솔직히 24토티은카 8조짜리랑 차이가 없음 걍 진짜 좋음..',
 '금카 사용중 CAP보다는 확실히 우위의 22시즌 나머지 120오버롤 대비해서 다 거기서 거기이고 이게 제일 가겨대비 성능도 좋음 딱 윙어에 10조 이상 금카 흥민이 쓰는건 조금 낭비임 이걸로 솔직히 흥민이는 FC온라인 종결임',
 ' 24토티+5 쓰다가 급여 때문에 이거 금카로 넘어왔는데 솔직히 블라인드 테스트하면 못 맞출 것 같음 비슷비슷한 듯',
 '3년째 챔스거주중 키보드유저 캡금이랑 다를거없고 몸싸움이랑 드리블은 여전히 너무 안좋다 바로 이적시장침투',
 '금카 한번 써보고 싶습니다 부탁드립니다 형님들!',
 '다른시즌에 비해서는 모르겠는데 22챔 금카 공미로두고 중거리 개맛있게먹는중 궤적이 진짜다름 금카 개좋음진짜 윗시즌금카부터는 14조라서 이게 가성비 국밥임',
 '금카 하한 전재산 제발팔아주세요ㅕ',
 '금카 상한 알박 3일째 제발 팔아주세요,.,,',
 '이거 금카 캐미받고 톱 어떤가요',
 '** 삭제된 글입니다 **',
 '현역케미가 없는데유',
 '요 며칠간 24토티 손흥민 은카 사용해보니까 23토티, 22챔스 8카보다 위치선정, 침투가 좋았음  참여도 3/3이 아니라 3/2인 느낌임  인벤 데이터 베이스를 어떻게 분석하는 지는 모르겠지만 본가 손흥민은 참여도 3/2임   22 챔스 금카 손흥민 , 24토티 손흥민 둘다 사용해본 결과 24토티 손흥민 5카로 정착함  이유는 24토티가 확실히 똑똑했고 골 포인트가 더 높았고 1대1 상황에서 24토티가 더 안정적임  어시스트 횟수도 24토티 손흥민 5카가 확실히 많았음  24토티는 st , cf , lw , rw로 사용했을때 패마로 특성단 22챔 금카보다 연계가 좋아서 포메를 다양하게 사용하기 좋았음   기존 손흥민 카드들이랑 비교했을 때 기존 성능 업그레이드는 당연하고 몸싸움이 진짜 많이 좋